# 02 — Preprocessing Pipeline

This notebook demonstrates the **feature engineering pipeline** implemented in `FeatureEngineer`.  
The pipeline transforms raw 5-sensor time-series into compact [0, π]-normalised feature vectors suitable for parameterised quantum circuits:

```
Raw 5-sensor → Sliding Windows (128) → 4 Stats/sensor (20D) → PCA(6) → [0, π] Scaling
```

## 1. Imports

We use `ThreeWLoader` to load a sample instance, then `FeatureEngineer` to process it through the full pipeline.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qml.samarone_junior.loaders import ThreeWLoader, FeatureEngineer

loader = ThreeWLoader(data_path="../../data/samarone_junior/3w")
fe = FeatureEngineer(n_components=6, window_size=128, stride=64)
print(f"Window size: {fe.window_size}, Stride: {fe.stride}, PCA components: {fe.n_components}")

## 2. Load Raw Data

We load a single normal (class 0) instance and extract just the 5 sensor columns as a NumPy array.

In [ ]:
instances = loader.list_instances(0)  # Normal class
if instances:
    df = pd.read_parquet(instances[0])
    raw = df[ThreeWLoader.SENSORS].dropna().values
    print(f"Raw shape: {raw.shape}  (timesteps × sensors)")
else:
    # Fallback: synthetic data for demonstration
    raw = np.random.randn(5000, 5)
    print(f"Using synthetic data: {raw.shape}")

## 3. Sliding Window Extraction

The first pipeline stage slides a fixed-size window (128 timesteps, stride 64) across the time series, producing overlapping segments.

In [ ]:
windows = fe.extract_windows(raw)
print(f"Windows shape: {windows.shape}  (n_windows × window_size × sensors)")
print(f"Number of windows: {windows.shape[0]}")

## 4. Statistical Feature Computation

For each window and each sensor, four statistics are computed: **mean**, **std**, **range**, and **kurtosis**.  
This yields a 20-dimensional feature vector per window (4 stats × 5 sensors).

In [ ]:
features = fe.compute_features(windows)
print(f"Feature matrix shape: {features.shape}  (n_windows × 20)")
print(f"Feature sample (first window): {features[0, :6].round(4)}")

## 5. PCA Reduction and [0, π] Scaling

PCA reduces the 20D features to 6 principal components, then MinMaxScaler maps them to [0, π] — the range expected by quantum angle-encoding gates.

In [ ]:
# Fit PCA + scaling on the already-computed features (from cell 8)
X_scaled = fe.fit_transform(features)
print(f"Scaled output shape: {X_scaled.shape}  (n_windows × {fe.n_components})")
print(f"Value range: [{X_scaled.min():.4f}, {X_scaled.max():.4f}] (should be [0, π ≈ 3.1416])")

In [ ]:
# Visualise the distribution of scaled features
fig, ax = plt.subplots(figsize=(8, 4))
for i in range(fe.n_components):
    ax.hist(X_scaled[:, i], bins=30, alpha=0.5, label=f"PC{i+1}")
ax.set_xlabel("Angle (radians)")
ax.set_ylabel("Count")
ax.set_title("Distribution of [0, π]-scaled PCA features")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6. Summary

The `FeatureEngineer` pipeline converts raw 5-sensor time series into compact 6D vectors in [0, π]:

1. **Sliding windows** (128 timesteps, stride 64) segment the time series.
2. **Statistical features** (mean, std, range, kurtosis) produce a 20D vector per window.
3. **PCA** reduces dimensionality from 20 → 6 components.
4. **MinMaxScaler** maps to [0, π] for quantum angle encoding.

These feature vectors feed directly into the Quantum Autoencoder circuits in the next notebook.